# Pilot Observability Inspection

Purpose  
Inspect Phase 1 pilot data to understand what can be observed, for how long, and with what stability.

Scope  
- Read-only inspection of local/raw_jsonl
- No transformations
- No cadence or Phase 2 decisions

In [3]:
from pathlib import Path
import re
import json
from datetime import datetime, timezone
import pandas as pd

# Locate repo root by finding local/raw_jsonl above the current working directory.
here = Path.cwd().resolve()
REPO_ROOT = None
for p in [here, *here.parents[:6]]:
    if (p / "local" / "raw_jsonl").exists():
        REPO_ROOT = p
        break

assert REPO_ROOT is not None, f"Could not find local/raw_jsonl above {here}"
RAW_DIR = REPO_ROOT / "local" / "raw_jsonl"

FILES = sorted(RAW_DIR.glob("*.jsonl"))
print("cwd:", here)
print("repo_root:", REPO_ROOT)
print("jsonl_file_count:", len(FILES))

# Pilot filename contract (Phase 1):
# {run_id}_{surface}_limit{limit}.jsonl
# Example: 20260105_233930_new_limit1000.jsonl
FILENAME_RE = re.compile(
    r"^(?P<run_id>\d{8}_\d{6})_(?P<surface>new|hot|rising|controversial)_limit(?P<limit>\d+)\.jsonl$"
)

def read_first_row(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.loads(next(f))

def parse_captured_utc(s: str) -> datetime:
    return datetime.fromisoformat(s.replace("Z", "+00:00")).astimezone(timezone.utc)

cwd: /Users/buddy/projects/reddit-early-dynamics/src/analysis/notebooks
repo_root: /Users/buddy/projects/reddit-early-dynamics
jsonl_file_count: 24


## Data inventory

In [4]:
# Data inventory:
# Build a file-level view of the pilot data by parsing filenames and counting rows.
# This confirms which runs and surfaces exist and that files are non-empty.

rows = []

for p in FILES:
    m = FILENAME_RE.match(p.name)
    if not m:
        raise ValueError(f"Unexpected filename format: {p.name}")

    row_count = sum(1 for _ in p.open("r", encoding="utf-8"))

    rows.append(
        {
            "run_id": m.group("run_id"),
            "surface": m.group("surface"),
            "limit": int(m.group("limit")),
            "path": str(p),
            "row_count": row_count,
        }
    )

inventory = (
    pd.DataFrame(rows)
    .sort_values(["run_id", "surface"])
    .reset_index(drop=True)
)

print("total_files:", len(inventory))
inventory.head(5)

total_files: 24


,run_id,surface,limit,path,row_count
0,20260105_233930,controversial,50,/Users/buddy/projects/reddit-early-dynamics/lo...,50
1,20260105_233930,hot,50,/Users/buddy/projects/reddit-early-dynamics/lo...,50
2,20260105_233930,new,1000,/Users/buddy/projects/reddit-early-dynamics/lo...,990
3,20260105_233930,rising,50,/Users/buddy/projects/reddit-early-dynamics/lo...,26
4,20260106_000029,controversial,50,/Users/buddy/projects/reddit-early-dynamics/lo...,50


In [5]:
# Check surface coverage per run_id and identify any missing surfaces.

expected_surfaces = {"new", "hot", "rising", "controversial"}

per_run = (
    inventory.groupby("run_id")["surface"]
    .apply(lambda s: sorted(set(s)))
    .rename("surfaces_present")
    .reset_index()
)

per_run["surfaces_missing"] = per_run["surfaces_present"].apply(
    lambda s: sorted(expected_surfaces - set(s))
)

per_run

,run_id,surfaces_present,surfaces_missing
0,20260105_233930,"[controversial, hot, new, rising]",[]
1,20260106_000029,"[controversial, hot, new, rising]",[]
2,20260106_002506,"[controversial, hot, new, rising]",[]
3,20260106_010039,"[controversial, hot, new, rising]",[]
4,20260106_020050,"[controversial, hot, new, rising]",[]
5,20260106_035233,"[controversial, hot, new, rising]",[]


In [6]:
# Confirm there are no empty files in the pilot data.
empty_files = inventory.loc[inventory["row_count"] == 0, ["run_id", "surface", "path"]]
empty_files

,run_id,surface,path


**Data inventory summary**

- 24 JSONL files
- 6 runs
- 4 surfaces per run: new, hot, rising, controversial
- No empty files
- new: ~985–990 rows
- hot: 50 rows
- controversial: 50 rows
- rising: 25–26 rows

## Snapshot timing

In [7]:
# Extract captured_utc from the first row of each surface file.
# This represents when each listing snapshot was taken.
timing_rows = []

for p in FILES:
    m = FILENAME_RE.match(p.name)
    row0 = read_first_row(p)

    timing_rows.append(
        {
            "run_id": m.group("run_id"),
            "surface": m.group("surface"),
            "captured_utc": parse_captured_utc(row0["captured_utc"]),
            "listing_run_id_in_row": row0.get("listing_run_id"),
        }
    )

timing = (
    pd.DataFrame(timing_rows)
    .sort_values(["run_id", "surface"])
    .reset_index(drop=True)
)

print("timing_rows:", len(timing))
timing.head(5)

timing_rows: 24


,run_id,surface,captured_utc,listing_run_id_in_row
0,20260105_233930,controversial,2026-01-05 23:39:37.825867+00:00,20260105_233930
1,20260105_233930,hot,2026-01-05 23:39:37.157475+00:00,20260105_233930
2,20260105_233930,new,2026-01-05 23:39:30.196947+00:00,20260105_233930
3,20260105_233930,rising,2026-01-05 23:39:37.618391+00:00,20260105_233930
4,20260106_000029,controversial,2026-01-06 00:00:38.470892+00:00,20260106_000029


In [8]:
# Measure how far apart surface snapshots are within the same run_id.
within_run = (
    timing.groupby("run_id")
    .agg(
        captured_min_utc=("captured_utc", "min"),
        captured_max_utc=("captured_utc", "max"),
        surfaces_present=("surface", lambda s: sorted(set(s))),
    )
    .reset_index()
    .sort_values("run_id")
)

within_run["within_run_spread_seconds"] = (
    (within_run["captured_max_utc"] - within_run["captured_min_utc"]).dt.total_seconds()
)

within_run[
    [
        "run_id",
        "captured_min_utc",
        "captured_max_utc",
        "within_run_spread_seconds",
        "surfaces_present",
    ]
]

,run_id,captured_min_utc,captured_max_utc,within_run_spread_seconds,surfaces_present
0,20260105_233930,2026-01-05 23:39:30.196947+00:00,2026-01-05 23:39:37.825867+00:00,7.628920,"[controversial, hot, new, rising]"
1,20260106_000029,2026-01-06 00:00:29.160477+00:00,2026-01-06 00:00:38.470892+00:00,9.310415,"[controversial, hot, new, rising]"
2,20260106_002506,2026-01-06 00:25:06.960746+00:00,2026-01-06 00:25:16.460620+00:00,9.499874,"[controversial, hot, new, rising]"
3,20260106_010039,2026-01-06 01:00:39.494451+00:00,2026-01-06 01:00:49.356414+00:00,9.861963,"[controversial, hot, new, rising]"
4,20260106_020050,2026-01-06 02:00:50.515194+00:00,2026-01-06 02:00:58.450860+00:00,7.935666,"[controversial, hot, new, rising]"
5,20260106_035233,2026-01-06 03:52:33.562168+00:00,2026-01-06 03:52:43.002903+00:00,9.440735,"[controversial, hot, new, rising]"


In [9]:
# Compute time gaps between consecutive runs using /new snapshots only.
new_times = (
    timing[timing["surface"] == "new"]
    .sort_values("captured_utc")
    .reset_index(drop=True)
)

new_times["gap_from_prev_minutes"] = (
    new_times["captured_utc"].diff().dt.total_seconds() / 60
)

new_times[["run_id", "captured_utc", "gap_from_prev_minutes"]]

,run_id,captured_utc,gap_from_prev_minutes
0,20260105_233930,2026-01-05 23:39:30.196947+00:00,NaN
1,20260106_000029,2026-01-06 00:00:29.160477+00:00,20.982726
2,20260106_002506,2026-01-06 00:25:06.960746+00:00,24.630004
3,20260106_010039,2026-01-06 01:00:39.494451+00:00,35.542228
4,20260106_020050,2026-01-06 02:00:50.515194+00:00,60.183679
5,20260106_035233,2026-01-06 03:52:33.562168+00:00,111.717450


**Snapshot timing summary**

- All runs include all four surfaces
- Within-run capture spread: ~7–10 seconds
- Run-to-run gaps range from ~21 to ~112 minutes
- No duplicate run timestamps

## Schema consistency

In [10]:
# Compare field counts across all pilot files to check for schema stability.
key_counts = []

for p in FILES:
    row0 = read_first_row(p)
    key_counts.append(
        {
            "file": p.name,
            "key_count": len(row0.keys()),
        }
    )

key_counts_df = pd.DataFrame(key_counts)

key_counts_df["key_count"].value_counts().sort_index()

key_count
112    19
113     5
Name: count, dtype: int64

In [11]:
# Identify which fields differ when key counts are not identical.
# Use one representative example for comparison.

# Pick one file with the smaller schema and one with the larger schema
small_schema_file = key_counts_df.loc[key_counts_df["key_count"].idxmin(), "file"]
large_schema_file = key_counts_df.loc[key_counts_df["key_count"].idxmax(), "file"]

small_row = read_first_row(RAW_DIR / small_schema_file)
large_row = read_first_row(RAW_DIR / large_schema_file)

small_keys = set(small_row.keys())
large_keys = set(large_row.keys())

{
    "only_in_larger_schema": sorted(large_keys - small_keys),
    "only_in_smaller_schema": sorted(small_keys - large_keys),
}

{'only_in_larger_schema': ['author_cakeday'], 'only_in_smaller_schema': []}

In [12]:
# Confirm required fields exist and behave as expected.

required_fields = [
    "id",
    "created_utc",
    "captured_utc",
    "surface",
    "listing_run_id",
]

missing_required = []

for p in FILES:
    row0 = read_first_row(p)
    missing = [f for f in required_fields if f not in row0]
    if missing:
        missing_required.append(
            {
                "file": p.name,
                "missing_fields": missing,
            }
        )

pd.DataFrame(missing_required)

""


In [13]:
# Check surface_rank behavior:
# - present for ranked surfaces
# - null for /new

rank_check = []

for p in FILES:
    row0 = read_first_row(p)
    rank_check.append(
        {
            "file": p.name,
            "surface": row0.get("surface"),
            "surface_rank_is_null": row0.get("surface_rank") is None,
        }
    )

rank_check_df = pd.DataFrame(rank_check)

rank_check_df.groupby(["surface", "surface_rank_is_null"]).size().reset_index(name="count")

,surface,surface_rank_is_null,count
0,controversial,False,6
1,hot,False,6
2,new,True,6
3,rising,False,6


**Schema consistency summary**

- Two schema sizes observed: 112 and 113 fields
- Only additive difference: author_cakeday
- No fields disappear across runs
- Required fields always present
- surface_rank null for new, present for ranked surfaces

## /new observability

In [14]:
# Load /new rows and extract created and captured timestamps
new_rows = []

for p in FILES:
    m = FILENAME_RE.match(p.name)
    if m.group("surface") != "new":
        continue

    with p.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            new_rows.append(
                {
                    "run_id": m.group("run_id"),
                    "id": row["id"],
                    "created_utc": datetime.fromtimestamp(
                        row["created_utc"], tz=timezone.utc
                    ),
                    "captured_utc": parse_captured_utc(row["captured_utc"]),
                }
            )

new_df = pd.DataFrame(new_rows)

print("new_rows:", len(new_df))
print("new_unique_posts:", new_df["id"].nunique())
print("columns:", new_df.columns.tolist())

new_rows: 5923
new_unique_posts: 1585
columns: ['run_id', 'id', 'created_utc', 'captured_utc']


In [15]:
# Compute post age at time of capture
new_df["age_seconds"] = (
    new_df["captured_utc"] - new_df["created_utc"]
).dt.total_seconds()

{
    "min_hours": float(new_df["age_seconds"].min() / 3600),
    "median_hours": float(new_df["age_seconds"].median() / 3600),
    "p90_hours": float(new_df["age_seconds"].quantile(0.9) / 3600),
    "max_hours": float(new_df["age_seconds"].max() / 3600),
}

{'min_hours': 0.002766873888888889,
 'median_hours': 3.36237838,
 'p90_hours': 5.878896571166667,
 'max_hours': 7.190989491111111}

In [16]:
# Count how many times each post appears in /new
snapshots_per_post = new_df.groupby("id").size()

{
    "unique_posts": int(snapshots_per_post.shape[0]),
    "min_snapshots": int(snapshots_per_post.min()),
    "median_snapshots": float(snapshots_per_post.median()),
    "max_snapshots": int(snapshots_per_post.max()),
}

{'unique_posts': 1585,
 'min_snapshots': 1,
 'median_snapshots': 4.0,
 'max_snapshots': 6}

In [17]:
# Compute first and last observation per post
per_post_span = (
    new_df.groupby("id")
    .agg(
        first_seen=("captured_utc", "min"),
        last_seen=("captured_utc", "max"),
    )
    .reset_index()
)

per_post_span["observable_span_minutes"] = (
    (per_post_span["last_seen"] - per_post_span["first_seen"])
    .dt.total_seconds()
    / 60
)

{
    "min_minutes": float(per_post_span["observable_span_minutes"].min()),
    "median_minutes": float(per_post_span["observable_span_minutes"].median()),
    "max_minutes": float(per_post_span["observable_span_minutes"].max()),
}

{'min_minutes': 0.0,
 'median_minutes': 141.33863745,
 'max_minutes': 253.05608701666668}

**/new observability summary**

- 5,923 total /new rows across all runs
- 1,585 unique posts observed
- Post age at capture:
  - minimum ≈ 0.003 hours
  - median ≈ 3.36 hours
  - 90th percentile ≈ 5.88 hours
  - maximum ≈ 7.19 hours
- Snapshots per post:
  - minimum 1
  - median 4
  - maximum 6
- Observable span per post (within pilot window):
  - minimum 0 minutes
  - median ≈ 141 minutes
  - maximum ≈ 253 minutes

## Overlap between pulls

In [18]:
# Build per-run /new ID sets and timestamps
new_runs = []

for p in FILES:
    m = FILENAME_RE.match(p.name)
    if m.group("surface") != "new":
        continue

    row0 = read_first_row(p)
    run_ts = parse_captured_utc(row0["captured_utc"])

    ids = set()
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            ids.add(row["id"])

    new_runs.append(
        {
            "run_id": m.group("run_id"),
            "captured_utc": run_ts,
            "n_ids": len(ids),
            "ids": ids,
        }
    )

new_runs_df = (
    pd.DataFrame(new_runs)
    .sort_values("captured_utc")
    .reset_index(drop=True)
)

new_runs_df[["run_id", "captured_utc", "n_ids"]]

,run_id,captured_utc,n_ids
0,20260105_233930,2026-01-05 23:39:30.196947+00:00,990
1,20260106_000029,2026-01-06 00:00:29.160477+00:00,990
2,20260106_002506,2026-01-06 00:25:06.960746+00:00,987
3,20260106_010039,2026-01-06 01:00:39.494451+00:00,987
4,20260106_020050,2026-01-06 02:00:50.515194+00:00,984
5,20260106_035233,2026-01-06 03:52:33.562168+00:00,985


In [19]:
# Consecutive-run overlap with time gap context
overlap_rows = []

for i in range(1, len(new_runs_df)):
    prev = new_runs_df.iloc[i - 1]
    curr = new_runs_df.iloc[i]

    overlap = len(prev["ids"] & curr["ids"])
    denom = min(prev["n_ids"], curr["n_ids"])
    overlap_pct = overlap / denom if denom else None

    gap_minutes = (curr["captured_utc"] - prev["captured_utc"]).total_seconds() / 60

    overlap_rows.append(
        {
            "run_id_prev": prev["run_id"],
            "run_id_curr": curr["run_id"],
            "gap_minutes": gap_minutes,
            "prev_n": prev["n_ids"],
            "curr_n": curr["n_ids"],
            "overlap_count": overlap,
            "overlap_pct": overlap_pct,
        }
    )

overlap_df = pd.DataFrame(overlap_rows)
overlap_df

,run_id_prev,run_id_curr,gap_minutes,prev_n,curr_n,overlap_count,overlap_pct
0,20260105_233930,20260106_000029,20.982726,990,990,942,0.951515
1,20260106_000029,20260106_002506,24.630004,990,987,930,0.942249
2,20260106_002506,20260106_010039,35.542228,987,987,901,0.912867
3,20260106_010039,20260106_020050,60.183679,987,984,832,0.845528
4,20260106_020050,20260106_035233,111.717450,984,985,733,0.744919


In [20]:
# Compact view with rounded percentages
view = overlap_df.copy()
view["gap_minutes"] = view["gap_minutes"].round(1)
view["overlap_pct"] = (view["overlap_pct"] * 100).round(2)

view[["run_id_prev", "run_id_curr", "gap_minutes", "overlap_count", "overlap_pct"]]

,run_id_prev,run_id_curr,gap_minutes,overlap_count,overlap_pct
0,20260105_233930,20260106_000029,21.0,942,95.15
1,20260106_000029,20260106_002506,24.6,930,94.22
2,20260106_002506,20260106_010039,35.5,901,91.29
3,20260106_010039,20260106_020050,60.2,832,84.55
4,20260106_020050,20260106_035233,111.7,733,74.49


**Overlap summary**

- 5 consecutive run pairs compared
- Time gaps between consecutive /new pulls range from ~21 minutes to ~112 minutes
- Overlap counts and percentages by gap:
  - ~21.0 min gap: 942 overlapping posts (95.15%)
  - ~24.6 min gap: 930 overlapping posts (94.22%)
  - ~35.5 min gap: 901 overlapping posts (91.29%)
  - ~60.2 min gap: 832 overlapping posts (84.55%)
  - ~111.7 min gap: 733 overlapping posts (74.49%)
- Overlap percentage declines monotonically as the gap between pulls increases